# Application Scenario 2: Training Dataset Creation

This scenario exports the output to the `output` directory.\
Please clear this directory between pipeline executions.

## Download the resources

- configuration file
- input directory containing the VRT file and the GeoPackage

In [ ]:
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/2/config.yaml -o config.yaml
!mkdir output
!mkdir input
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/2/input/362000_5714500.tiff -o input/362000_5714500.tiff
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/2/input/362000_5715000.tiff -o input/362000_5715000.tiff
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/2/input/362000_5715500.tiff -o input/362000_5715500.tiff
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/2/input/362000_5716000.tiff -o input/362000_5716000.tiff
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/2/input/362500_5714500.tiff -o input/362500_5714500.tiff
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/2/input/362500_5715000.tiff -o input/362500_5715000.tiff
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/2/input/362500_5715500.tiff -o input/362500_5715500.tiff
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/2/input/362500_5716000.tiff -o input/362500_5716000.tiff
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/2/input/363000_5714500.tiff -o input/363000_5714500.tiff
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/2/input/363000_5715000.tiff -o input/363000_5715000.tiff
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/2/input/363000_5715500.tiff -o input/363000_5715500.tiff
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/2/input/363000_5716000.tiff -o input/363000_5716000.tiff
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/2/input/363500_5714500.tiff -o input/363500_5714500.tiff
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/2/input/363500_5715000.tiff -o input/363500_5715000.tiff
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/2/input/363500_5715500.tiff -o input/363500_5715500.tiff
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/2/input/363500_5716000.tiff -o input/363500_5716000.tiff
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/2/input/building_footprints.gpkg -o input/building_footprints.gpkg
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/2/input/rgb.vrt -o input/rgb.vrt

## Installation

### With `pip`

In [ ]:
!pip install geospaitial-lab-aviary[cli] ipywidgets

### With `uv`

In [ ]:
!uv pip install geospaitial-lab-aviary[cli] ipywidgets

---

## Python API

### Imports

In [ ]:
import warnings
from pathlib import Path

import aviary
import aviary.pipeline
import aviary.tile
import aviary.utils

### Check the version

In [ ]:
print(aviary.__version__)

### Suppress experimental warnings

Some components used in this scenario are experimental.

In [ ]:
warnings.filterwarnings('ignore', category=aviary.AviaryExperimentalWarning)

### Define the `Grid`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/core/grid)

In [ ]:
bounding_box = aviary.BoundingBox(
    x_min=362000,
    y_min=5714500,
    x_max=364000,
    y_max=5716500,
)

grid = aviary.Grid.from_bounding_box(
    bounding_box=bounding_box,
    tile_size=500,
)

### Define the `TileFetcher`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/tile/tile_fetcher/tile_fetcher)

In [ ]:
vrt_fetcher = aviary.tile.VRTFetcher(
    path=Path('input/rgb.vrt'),
    epsg_code=25832,
    channel_names=['r', 'g', 'b'],
    tile_size=500,
    ground_sampling_distance=.2,
)

gpkg_fetcher = aviary.tile.GPKGFetcher(
    path=Path('input/building_footprints.gpkg'),
    epsg_code=25832,
    channel_name='building_footprints',
    tile_size=500,
)

composite_fetcher = aviary.tile.CompositeFetcher(
    tile_fetchers=[vrt_fetcher, gpkg_fetcher],
)

### Define the `TilesProcessor`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/tile/tiles_processor/tiles_processor)

In [ ]:
raster_exporter_rgb = aviary.tile.RasterExporter(
    channel_names=['r', 'g', 'b'],
    epsg_code=25832,
    path=Path('output/rgb'),
)

rasterize_processor = aviary.tile.RasterizeProcessor(
    channel_name='building_footprints',
    ground_sampling_distance=.2,
    foreground_value=1,
    background_value=0,
    dtype=aviary.DType.UINT8,
)

raster_exporter_building_footprints = aviary.tile.RasterExporter(
    channel_names='building_footprints',
    epsg_code=25832,
    path=Path('output/building_footprints'),
)

composite_processor = aviary.tile.SequentialCompositeProcessor(
    tiles_processors=[
        raster_exporter_rgb,
        rasterize_processor,
        raster_exporter_building_footprints,
    ],
)

### Define the `Logger`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/utils/logger)

In [ ]:
logger = aviary.utils.Logger(
    sink='output/application_scenario_2.log',
    level=aviary.LogLevel.TRACE,
)

### Define the `TilePipeline`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/pipeline/pipeline)

In [ ]:
tile_pipeline = aviary.pipeline.TilePipeline(
    grid=grid,
    tile_fetcher=composite_fetcher,
    tiles_processor=composite_processor,
    tile_loader_batch_size=4,
    tile_loader_max_num_threads=None,
    tile_loader_num_prefetched_tiles=1,
    show_progress=True,
)

### Execute the `TilePipeline`

In [ ]:
tile_pipeline()

---

## CLI

### Check the version

In [ ]:
!aviary --version

### Suppress experimental warnings

Some components used in this scenario are experimental.

In [ ]:
%env AVIARY_EXPERIMENTAL_WARNINGS=false

### Validate the configuration file

[Docs](https://geospaitial-lab.github.io/aviary/cli_reference/aviary_pipeline/pipeline_validate)

In [ ]:
!aviary pipeline validate config.yaml

### Set the path to the log file and the log level

In [ ]:
%env AVIARY_LOG_PATH=output/application_scenario_2.log
%env AVIARY_LOG_LEVEL=trace

### Execute the `TilePipeline`

[Docs](https://geospaitial-lab.github.io/aviary/cli_reference/aviary_pipeline/pipeline_run)

In [ ]:
!aviary pipeline run config.yaml